# 1. DATA CLEANING 

## Objective

The purpose of this phase is to assess, clean, standardize, and validate all source datasets before integrating them into a unified analytical dataset.

The cleaning process focuses on:

- Missing value handling
- Duplicate detection
- Data type corrections
- Outlier identification
- Standardization of categorical values
- Validation of healthcare measurements
- Referential integrity assessment

The cleaned datasets generated in this phase will be used for statistical analysis, exploratory data analysis, machine learning, clustering, NLP, and Power BI reporting.

In [ ]:
# ============================================
# Import Required Libraries
# ============================================

from pathlib import Path

conflicts = [
    path for path in Path.cwd().iterdir()
    if path.name.lower() in {"pandas.py", "pandas"}
]
if conflicts:
    raise ImportError(f"Rename or remove this local pandas conflict: {conflicts}")

import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# ============================================
# Load Datasets
# ============================================

member = pd.read_csv("Dim_Member.csv")
profile = pd.read_csv("Dim_Member_Profile.csv")
wellness = pd.read_csv("Fact_Wellness_Program.csv")
clinical = pd.read_csv("Fact_Clinical_Screenings.csv")
claims = pd.read_csv("Fact_Claims_Utilization.csv")
feedback = pd.read_csv("Fact_Member_Feedback.csv")

print("Files Loaded Successfully")

## 1.1 Initial dataset Inspection

In [ ]:
datasets = {
    "Member": member,
    "Profile": profile,
    "Wellness": wellness,
    "Clinical": clinical,
    "Claims": claims,
    "Feedback": feedback
}

for name, df in datasets.items():

    print("="*80)
    print(name.upper())
    print("="*80)

    print("\nShape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nSample Data:")
    display(df.head())

## 1.2 Data Quality Summary

In [ ]:
quality_summary = []

for name, df in datasets.items():

    quality_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isna().sum().sum(),
        "Duplicate Rows": df.duplicated().sum()
    })

quality_summary = pd.DataFrame(quality_summary)
quality_summary

## 1.3 Missing Value Assessment

In [ ]:
for name, df in datasets.items():

    print("\n" + "="*80)
    print(f"{name.upper()} - Missing Values")
    print("="*80)

    missing_df = pd.DataFrame({
        "Column": df.columns,
        "Missing_Count": df.isnull().sum().values,
        "Missing_Percent": np.round((df.isnull().sum().values / len(df))*100, 2)
    })

    display(missing_df.sort_values("Missing_Count", ascending=False))

## 1.4 Duplicate Record Detection

In [ ]:
for name, df in datasets.items():

    print(f"\n{name}")
    dup_count = df.duplicated().sum()
    print("Duplicate Rows:", dup_count)

## 1.5 Numeric Variable Profiling

In [ ]:
for name, df in datasets.items():

    print("\n" + "="*80)
    print(name.upper())
    print("="*80)

    numeric_cols = df.select_dtypes(include=np.number).columns

    if len(numeric_cols) > 0: 
        display(df[numeric_cols].describe())

## 1.6 Categorical Variable Profiling

In [ ]:
for name, df in datasets.items():

    print("\n" + "="*80)
    print(name.upper())
    print("="*80)

    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols[:10]:
        print(f"\nColumn : {col}")
        display(df[col].value_counts(dropna=False).head(15))

## 1.7 Creating Working Copies For Cleaning
Original datasets will be preserved.

In [ ]:
member_clean = member.copy()
profile_clean = profile.copy()
wellness_clean = wellness.copy()
clinical_clean = clinical.copy()
claims_clean = claims.copy()
feedback_clean = feedback.copy()

In [ ]:
# ============================================
# Standardize Member IDs
# ============================================

def clean_member_id(series):
    return (series.astype(str).str.upper().str.strip().str.replace("-", "", regex=False).str.replace(" ", "", regex=False))


for df in [
    member_clean,
    profile_clean,
    wellness_clean,
    clinical_clean,
    claims_clean,
    feedback_clean
]:

    if "Member_ID" in df.columns:
        df["Member_ID"] = clean_member_id(df["Member_ID"])

In [ ]:
member_clean = member_clean.drop_duplicates()
profile_clean = profile_clean.drop_duplicates()
wellness_clean = wellness_clean.drop_duplicates()
clinical_clean = clinical_clean.drop_duplicates()
claims_clean = claims_clean.drop_duplicates()
feedback_clean = feedback_clean.drop_duplicates()

## 1.8 Create Data Cleaning Audit Report

In [ ]:
cleaning_audit = []

tables = {
    "Member": member_clean,
    "Profile": profile_clean,
    "Wellness": wellness_clean,
    "Clinical": clinical_clean,
    "Claims": claims_clean,
    "Feedback": feedback_clean
}

for name, df in tables.items():
    cleaning_audit.append({
        "Dataset": name,
        "Rows_After_Cleaning":len(df),
        "Missing_Values":df.isna().sum().sum(),
        "Duplicate_Rows":df.duplicated().sum()
    })

cleaning_audit = pd.DataFrame(cleaning_audit)
cleaning_audit

In [ ]:
#==========================
# Export Cleaned Datasets
#=========================
member_clean.to_csv("clean_member.csv",index=False)
profile_clean.to_csv("clean_profile.csv",index=False)
wellness_clean.to_csv("clean_wellness.csv",index=False)
clinical_clean.to_csv("clean_clinical.csv",index=False)
claims_clean.to_csv("clean_claims.csv",index=False)
feedback_clean.to_csv("clean_feedback.csv",index=False)
print("Clean datasets exported successfully.")

# 2. DATA INTEGRATION & MASTER DATA CREATION 

### Objective

The cleaned source datasets are integrated to create a unified member-level analytical dataset.

This master dataset combines:

- Demographic information
- Member profile information
- Wellness participation
- Clinical indicators
- Claims utilization

The resulting dataset will serve as the primary source for:

- Statistical Analysis
- Exploratory Data Analysis
- Feature Engineering
- Machine Learning
- Clustering
- Power BI Reporting

Member feedback data will be maintained separately for NLP analysis.

In [ ]:
print("Member Dataset :", member_clean["Member_ID"].nunique())
print("Profile Dataset :", profile_clean["Member_ID"].nunique())
print("Wellness Dataset :", wellness_clean["Member_ID"].nunique())
print("Clinical Dataset :", clinical_clean["Member_ID"].nunique())
print("Claims Dataset :", claims_clean["Member_ID"].nunique())
print("Feedback Dataset :", feedback_clean["Member_ID"].nunique())

In [ ]:
#==================================================================
# Checking referential integrity between Member and other datasets
#==================================================================

member_ids = set(member_clean["Member_ID"])

tables = {
    "Profile": profile_clean,
    "Wellness": wellness_clean,
    "Clinical": clinical_clean,
    "Claims": claims_clean,
    "Feedback": feedback_clean
}

for name, df in tables.items():
    orphan_records = df[~df["Member_ID"].isin(member_ids)]
    print(f"{name} Orphan Records: {len(orphan_records)}")

In [ ]:
#===============================================================
# Create master dataset by merging all datasets on Member_ID
#===============================================================

master_df = (
    member_clean.merge(profile_clean, on="Member_ID", how="left")
    .merge(wellness_clean, on="Member_ID", how="left")
    .merge(clinical_clean, on="Member_ID", how="left")
    .merge(claims_clean, on="Member_ID", how="left")
)

In [ ]:
print(master_df.shape)
master_df.info()

In [ ]:
missing_summary = pd.DataFrame({

    "Column": master_df.columns,
    "Missing_Count": master_df.isna().sum().values,
    "Missing_Percent": round(master_df.isna().mean()*100,2).values
})

missing_summary.sort_values("Missing_Percent", ascending=False)

In [ ]:
master_df.to_csv("Gold_Member_Health.csv", index=False)
print("Gold dataset exported successfully.")

### Data Integration Summary

The cleaned datasets were successfully integrated using Member_ID as the common business key.

The resulting Gold_Member_Health dataset provides a consolidated member-level analytical view combining:

- Demographics
- Member Profile Information
- Wellness Program Participation
- Clinical Indicators
- Claims Utilization Metrics

This dataset will serve as the primary input for all subsequent analytical and predictive tasks.

# 3. SQL ANALYSIS

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')

In [ ]:
master_df.to_sql("Gold_Member_Health", conn, index=False, if_exists="replace")

print("Dataset loaded into SQL successfully.")

In [ ]:
#===========================================
# member distribution by region
#===========================================
#Which regions have the largest member populations?

query = """
SELECT
    Region,
    COUNT(*) AS Total_Members
FROM Gold_Member_Health
GROUP BY Region
ORDER BY Total_Members DESC;
"""

region_population =pd.read_sql(query, conn)
region_population 

### Business Insight
Region X contains the largest enrolled member population.

This indicates a higher concentration of healthcare demand
and may require additional preventive care initiatives.

In [ ]:
#==========================================
# Screening participation rate by region
#==========================================
#Which regions are most engaged in preventive care?

query = """
SELECT
    Region, COUNT(*) AS Total_Records,
    AVG(Preventive_Screenings_Completed)*100
        AS Avg_Screenings
FROM Gold_Member_Health
GROUP BY Region
ORDER BY Avg_Screenings DESC;
"""

screening_region =pd.read_sql(query, conn)
screening_region 

### Business Insight
Regions with higher screening participation demonstrate stronger
preventive healthcare engagement.

Regions with lower participation should be targeted through
outreach programs and awareness campaigns.

In [ ]:
#===================================================
# High-risk members based on BMI and glucose levels
#===================================================
#Which members may require preventive intervention?

query = """
SELECT
    Member_ID,
    Age,
    BMI,
    Glucose, HbA1c, Systolic_BP,
    Smoking_Status
FROM Gold_Member_Health
WHERE BMI > 30 OR HbA1c > 6.5 OR Systolic_BP > 140
  OR Glucose >= 126;
"""

high_risk_members  = pd.read_sql(query, conn)

high_risk_members.head() 

### Business Insight
Members identified through elevated BMI, glucose,
HbA1c, or blood pressure levels may benefit from
early intervention programs.

These members represent the primary target group
for disease prevention initiatives.

In [ ]:
#===================================
# Cost Analysis by Region
#===================================
# Which regions generate the highest healthcare costs?

query = """
SELECT
    Region,
    ROUND(SUM(Costs),2) AS Total_Healthcare_Cost,
    ROUND(AVG(Costs),2) AS Average_Cost
FROM Gold_Member_Health
GROUP BY Region
ORDER BY Total_Healthcare_Cost DESC;
"""

cost_region = pd.read_sql(query, conn)

cost_region

### Business Insight
Regions generating higher healthcare costs should be
further investigated to determine whether increased
disease burden or healthcare utilization is driving costs.

In [ ]:
#===========================
# Cost by Age group
#===========================
# How does healthcare spending vary across age groups?

query = """
SELECT
CASE
    WHEN Age BETWEEN 18 AND 30 THEN '18-30'
    WHEN Age BETWEEN 31 AND 45 THEN '31-45'
    WHEN Age BETWEEN 46 AND 60 THEN '46-60'
    ELSE '60+'
END AS Age_Group,

ROUND(AVG(Costs),2) AS Average_Cost,
ROUND(SUM(Costs),2) AS Total_Cost

FROM Gold_Member_Health

GROUP BY Age_Group

ORDER BY Average_Cost DESC;
"""

cost_age = pd.read_sql(query, conn)
cost_age

### Business Insight
Healthcare costs generally increase with age due to
greater prevalence of chronic conditions and healthcare utilization.

In [ ]:
#===============================
# Wellness program Effectiveness 
#===============================
# Does participation reduce healthcare utilization / improves outcomes?

query = """
SELECT

Program_Enrolled_Flag,

ROUND(AVG(Costs),2) AS Avg_Cost,

ROUND(AVG(Visits),2) AS Avg_Visits,

ROUND(AVG(BMI),2) AS Avg_BMI

FROM Gold_Member_Health

GROUP BY Program_Enrolled_Flag;
"""

wellness_impact = pd.read_sql(query, conn)

wellness_impact

### Business Insight
Members enrolled in wellness programs may demonstrate
different healthcare utilization patterns compared
with non-participants.

This helps evaluate program effectiveness.

In [ ]:
#===============================
# Top high cost members
#===============================
# Which members generate the highest healthcare costs?

query = """
SELECT *
FROM (
    SELECT Member_ID, Region, Costs,
    RANK() OVER( ORDER BY Costs DESC) AS Cost_Rank

    FROM Gold_Member_Health
)

WHERE Cost_Rank <= 20;
"""

top_cost_members = pd.read_sql(query, conn)
top_cost_members

### Business Insight
A relatively small number of members often account
for a large proportion of total healthcare expenditure.

These members should be considered for care-management programs.

In [ ]:
#===============================
# Risk Categorization
#===============================
# How can members be grouped by risk?

query = """
SELECT Member_ID, BMI, Glucose,
CASE

WHEN BMI > 35 OR Glucose > 140
THEN 'High Risk'

WHEN BMI > 28 OR Glucose > 110
THEN 'Moderate Risk'

ELSE 'Low Risk'

END AS Risk_Category

FROM Gold_Member_Health;
"""

pd.read_sql(query, conn)

In [ ]:
#Which members have higher-than-average healthcare cost?

query = """
SELECT Member_ID, Costs
FROM Gold_Member_Health

WHERE Costs >
(
    SELECT AVG(Costs)
    FROM Gold_Member_Health
);
"""

above_avg_cost = pd.read_sql(query, conn)
above_avg_cost 

### Business insight 
Members with above-average healthcare costs may represent
higher-risk populations and should be investigated for
cost-reduction opportunities through preventive care programs.

In [ ]:
#=============================================
# Chronic Disease rate by region
#=============================================
# Which regions have the greatest future disease burden?

query = """
SELECT Region, COUNT(*) AS Total_Members,
SUM(Develops_Chronic_Disease_12M) AS Chronic_Cases,

ROUND( 100.0 * SUM(Develops_Chronic_Disease_12M) / COUNT(*),2) AS Chronic_Disease_Rate

FROM Gold_Member_Health
GROUP BY Region
ORDER BY Chronic_Disease_Rate DESC;
"""

chronic_region = pd.read_sql(query, conn)
chronic_region

# 3. STATISTICAL ANALYSIS
### Objective

Statistical analysis was performed to understand the characteristics of the member population, identify key healthcare trends, and validate business assumptions related to disease risk, healthcare utilization, and medical costs.

The results of this analysis will support exploratory data analysis, feature engineering, predictive modeling, and business recommendations.

In [ ]:
#===============================================
# Descriptive Statistics for Key health metrics
#===============================================


stats_columns = [
    "BMI",
    "Glucose",
    "Cholesterol",
    "Costs",
    "Visits"
]

master_df[stats_columns].describe().T

In [ ]:
numeric_data = master_df[stats_columns].apply(pd.to_numeric, errors="coerce")

summary_stats = pd.DataFrame({
    "Mean": numeric_data.mean(),
    "Median": numeric_data.median(),
    "Std_Dev": numeric_data.std(),
    "Minimum": numeric_data.min(),
    "Maximum": numeric_data.max()

})

summary_stats

#### Interpretation

The descriptive statistics provide an overview of the population's clinical indicators and healthcare utilization.

Variables exhibiting large standard deviations may indicate substantial variation across members and could require further investigation during exploratory analysis.

## 3.1 Correlation Analysis
Understanding relationships between different features helps explain disease progression and healthcare spending patterns.

In [ ]:
correlation_cols = [
    "BMI",
    "Glucose",
    "HbA1c",
    "Exercise_Frequency",
    "Costs",
    "Visits"
]

# Convert all correlation columns to numeric; bad values like "Unknown" become NaN
numeric_df = master_df[correlation_cols].apply(pd.to_numeric, errors="coerce")

# Compute correlation on numeric data only
corr_matrix = numeric_df.corr()

corr_matrix

## 3.2 Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Matrix")
plt.show()

## 3.3 Hypothesis testing

### Objective

Hypothesis testing is performed to statistically validate key business assumptions related to disease risk, preventive care effectiveness, wellness program impact, healthcare utilization, and healthcare costs.

The results of these tests will help determine whether observed patterns within the member population are statistically significant and can support future healthcare interventions and strategic decision-making.

Significance Level (α): 0.05

Decision Rule:

- If p-value < 0.05 → Reject Null Hypothesis
- If p-value ≥ 0.05 → Fail to Reject Null Hypothesis

In [ ]:
from scipy.stats import ttest_ind
from scipy.stats import pearsonr

In [ ]:
numeric_columns = [
   "Age",
    "BMI",
    "Glucose",
    "HbA1c",
    "Cholesterol",
    "LDL_Cholesterol",
    "HDL_Cholesterol",
    "Triglycerides",

    "Systolic_BP",
    "Diastolic_BP",

    "Exercise_Frequency",
    "Diet_Score",

    "Visits",
    "ER_Visits",
    "Hospital_Admissions",
    "Medications",

    "Costs",

    "Preventive_Screenings_Completed",
    "Health_Risk_Assessment_Completed",

    "Chronic_Condition_Count"

]

for col in numeric_columns:

    if col in master_df.columns:

        master_df[col] = pd.to_numeric(
            master_df[col],
            errors="coerce"
        )

## HYPOTHESIS 1

### Business Question

Does higher BMI increase the likelihood of developing chronic disease?

### Null Hypothesis (H0)

There is no significant difference in BMI between members who develop chronic disease and members who do not.

### Alternative Hypothesis (H1)

Members who develop chronic disease have significantly higher BMI values than members who do not.

In [ ]:
master_df["BMI"] = pd.to_numeric(
    master_df["BMI"],
    errors="coerce"
)

disease_yes = master_df[master_df["Develops_Chronic_Disease_12M"] == 1]["BMI"]

disease_no = master_df[master_df["Develops_Chronic_Disease_12M"] == 0]["BMI"]

t_stat, p1 = ttest_ind(
    disease_yes,
    disease_no,
    nan_policy="omit"
)

print("Hypothesis 1")
print("T Statistic :", round(t_stat,4))
print("P Value :", round(p1,4))

if p1 < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

### HYPOTHESIS 2

#### Business Question

Does participation in preventive health screenings reduce healthcare costs?

#### Null Hypothesis (H0)

There is no significant difference in healthcare costs between members who participate in screenings and those who do not.

#### Alternative Hypothesis (H1)

Members who participate in preventive screenings incur significantly lower healthcare costs.

In [ ]:
screening_yes = master_df[master_df["Screening_Participation"].astype(str).str.upper().isin(["YES","Y","TRUE","1"])]["Costs"]

screening_no = master_df[
    ~master_df["Screening_Participation"].astype(str)
    .str.upper()
    .isin(["YES","Y","TRUE","1"])
]["Costs"]

t_stat, p2 = ttest_ind(
    screening_yes,
    screening_no,
    nan_policy="omit"
)

print("Hypothesis 2")
print("T Statistic :", round(t_stat,4))
print("P Value :", round(p2,4))

if p2 < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

### HYPOTHESIS 3

#### Business Question

Do wellness program participants demonstrate lower healthcare utilization?

#### Null Hypothesis (H0)

There is no significant difference in healthcare utilization between wellness participants and non-participants.

#### Alternative Hypothesis (H1)

Members enrolled in wellness programs have significantly lower healthcare utilization.

In [ ]:
wellness_yes = master_df[master_df["Program_Enrolled_Flag"] == 1]["Visits"]

wellness_no = master_df[master_df["Program_Enrolled_Flag"] == 0]["Visits"]

t_stat, p3 = ttest_ind(
    wellness_yes,
    wellness_no,
    nan_policy="omit"
)

print("Hypothesis 3")
print("T Statistic :", round(t_stat,4))
print("P Value :", round(p3,4))

if p3 < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

### HYPOTHESIS 4

#### Business Question

Does exercise frequency influence healthcare costs?

#### Null Hypothesis (H0)

There is no relationship between exercise frequency and healthcare costs.

#### Alternative Hypothesis (H1)

Higher exercise frequency is associated with lower healthcare costs.

In [ ]:
exercise_data = master_df[["Exercise_Frequency","Costs"]].dropna()

corr, p4 = pearsonr(exercise_data["Exercise_Frequency"],exercise_data["Costs"])

print("Hypothesis 4")
print("Correlation :", round(corr,4))
print("P Value :", round(p4,4))

if p4 < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

### HYPOTHESIS 5

#### Business Question
Does family history of diabetes increase chronic disease risk?

#### Null Hypothesis (H0)
There is no significant relationship between family history of diabetes and future chronic disease development.

#### Alternative Hypothesis (H1)
Members with a family history of diabetes are significantly more likely to develop chronic disease.

In [ ]:
family_yes = master_df[
    master_df["Family_History_Diabetes"]
    .astype(str)
    .str.upper()
    .isin(["YES","Y","TRUE","1"])
]["Develops_Chronic_Disease_12M"]

family_no = master_df[
    ~master_df["Family_History_Diabetes"]
    .astype(str)
    .str.upper()
    .isin(["YES","Y","TRUE","1"])
]["Develops_Chronic_Disease_12M"]

t_stat, p5 = ttest_ind(
    family_yes,
    family_no,
    nan_policy="omit"
)

print("Hypothesis 5")
print("T Statistic :", round(t_stat,4))
print("P Value :", round(p5,4))

if p5 < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

### HYPOTHESIS 6

#### Business Question

Does elevated glucose level increase future chronic disease risk?

#### Null Hypothesis (H0)

There is no significant difference in glucose levels between members who develop chronic disease and those who do not.

#### Alternative Hypothesis (H1)

Members who develop chronic disease have significantly higher glucose levels.

In [ ]:
disease_yes = master_df[master_df["Develops_Chronic_Disease_12M"] == 1]["Glucose"]

disease_no = master_df[master_df["Develops_Chronic_Disease_12M"] == 0]["Glucose"]

t_stat, p6 = ttest_ind(
    disease_yes,
    disease_no,
    nan_policy="omit"
)

print("Hypothesis 6")
print("T Statistic :", round(t_stat,4))
print("P Value :", round(p6,4))

if p6 < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

# Hypothesis Testing Summary

| Hypothesis | Result | Business Interpretation |
|------------|---------|-------------------------|
| H1 | Reject/Fail H0 | BMI significantly influences chronic disease risk |
| H2 | Reject/Fail H0 | Screening participation impacts healthcare costs |
| H3 | Reject/Fail H0 | Wellness programs affect healthcare utilization |
| H4 | Reject/Fail H0 | Exercise frequency influences healthcare costs |
| H5 | Reject/Fail H0 | Family history contributes to disease risk |
| H6 | Reject/Fail H0 | Elevated glucose levels increase disease risk |

In [ ]:
hypothesis_summary = pd.DataFrame({

    "Hypothesis":[

        "H1 - BMI and Chronic Disease Risk",
        "H2 - Screening Participation and Costs",
        "H3 - Wellness Participation and Utilization",
        "H4 - Exercise Frequency and Costs",
        "H5 - Family History and Disease Risk",
        "H6 - Glucose and Disease Risk"

    ],

    "P_Value":[

        p1,
        p2,
        p3,
        p4,
        p5,
        p6

    ],

    "Decision":[

        "Reject H0" if p1 < 0.05 else "Fail to Reject H0",
        "Reject H0" if p2 < 0.05 else "Fail to Reject H0",
        "Reject H0" if p3 < 0.05 else "Fail to Reject H0",
        "Reject H0" if p4 < 0.05 else "Fail to Reject H0",
        "Reject H0" if p5 < 0.05 else "Fail to Reject H0",
        "Reject H0" if p6 < 0.05 else "Fail to Reject H0"

    ]
})

hypothesis_summary

In [ ]:
master_df["Smoking_Status"] = (
    master_df["Smoking_Status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Smoking_Status"] = master_df["Smoking_Status"].replace({

    "SMOKER": "Smoker",
    "CURRENT SMOKER": "Smoker",
    "Y": "Smoker",
    "YES": "Smoker",

    "NON-SMOKER": "Non-Smoker",
    "NON SMOKER": "Non-Smoker",
    "NONSMOKER": "Non-Smoker",
    "N": "Non-Smoker",
    "NO": "Non-Smoker",

    "?": "Unknown",
    "-": "Unknown",
    "UNKNOWN": "Unknown"

})

In [ ]:
# Age
master_df.loc[
    (master_df["Age"] < 18) |
    (master_df["Age"] > 100),
    "Age"
] = np.nan

# BMI
master_df.loc[
    (master_df["BMI"] < 10) |
    (master_df["BMI"] > 70),
    "BMI"
] = np.nan

# Glucose
master_df.loc[
    (master_df["Glucose"] < 40) |
    (master_df["Glucose"] > 600),
    "Glucose"
] = np.nan

# HbA1c
master_df.loc[
    (master_df["HbA1c"] < 3) |
    (master_df["HbA1c"] > 20),
    "HbA1c"
] = np.nan

# Cholestrol
master_df.loc[
    (master_df["Cholesterol"] < 50) |
    (master_df["Cholesterol"] > 500),
    "Cholesterol"
] = np.nan

# Exercise Frequency
master_df.loc[
    (master_df["Exercise_Frequency"] < 0) |
    (master_df["Exercise_Frequency"] > 7),
    "Exercise_Frequency"
] = np.nan


# Diet Score
master_df.loc[
    (master_df["Diet_Score"] < 0) |
    (master_df["Diet_Score"] > 100),
    "Diet_Score"
] = np.nan

util_cols = [

    "Visits",
    "ER_Visits",
    "Hospital_Admissions",
    "Medications",
    "Chronic_Condition_Count"

]

for col in util_cols:

    master_df.loc[
        master_df[col] < 0,
        col
    ] = np.nan


# costs
master_df.loc[
    master_df["Costs"] < 0,
    "Costs"
] = np.nan


# Systolic BP
master_df.loc[
    (master_df["Systolic_BP"] < 70) |
    (master_df["Systolic_BP"] > 250),
    "Systolic_BP"
] = np.nan


#visits
master_df.loc[
    (master_df["Visits"] < 0) |
    (master_df["Visits"] > 100),
    "Visits"
] = np.nan


# medications
master_df.loc[
    (master_df["Medications"] < 0) |
    (master_df["Medications"] > 30),
    "Medications"
] = np.nan


In [ ]:
master_df["Plan_Type"] = (
    master_df["Plan_Type"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Plan_Type"] = (
    master_df["Plan_Type"]
    .replace({

        "EMPLOYER GROUP":"Employer Group",
        "GROUP":"Employer Group",

        "MEDICARE ADV":"Medicare Advantage",
        "MA":"Medicare Advantage",
        "MEDICARE ADVANTAGE":"Medicare Advantage",

        "INDIVIDUAL & FAMILY":"Individual & Family",
        "INDIVIDUAL AND FAMILY":"Individual & Family",
        "IFP":"Individual & Family",

        "MEDICAID":"Medicaid",
        "MEDI-CAL":"Medicaid",

        "FEHB":"Federal Employee (FEHB)",
        "FEDERAL EMPLOYEE":"Federal Employee (FEHB)",
        "FEDERAL EMPLOYEE (FEHB)":"Federal Employee (FEHB)",

        "COMMERCIAL":"Commercial"

    })
)

In [ ]:
master_df["Program_Status"] = (
    master_df["Program_Status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Program_Status"] = (
    master_df["Program_Status"]
    .replace({

        "NOT ENROLLED":"Not Enrolled",
        "N/E":"Not Enrolled",

        "COMPLETE":"Completed",
        "COMPLETED":"Completed",

        "DROPPED":"Dropped",
        "DROPPED OUT":"Dropped",
        "WITHDRAWN":"Dropped",

        "ACTIVE":"Active",
        "IN PROGRESS":"Active"

    })
)

In [ ]:
master_df["Gender"] = (
    master_df["Gender"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Gender"] = (
    master_df["Gender"]
    .replace({

        "MALE":"Male",
        "FEMALE":"Female",
        "OTHER":"Other",
        "PREFER NOT TO SAY":"Prefer Not To Say",
        "M":"Male",
        "F": "Female",
        "NB" :"Other",
        "NON-BINARY/OTHER" :"Other",
        "NON-BINARY" : "Other",
        "X" : "Other"

    })
)

In [ ]:
master_df["Preferred_Language"] = (
    master_df["Preferred_Language"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Preferred_Language"] = (
    master_df["Preferred_Language"]
    .replace({

        # English
        "ENGLISH": "English",
        "ENG": "English",
        # Spanish
        "SPANISH": "Spanish",
        "SPAN.": "Spanish",
        "ESPANOL": "Spanish",
        # Chinese
        "CHINESE": "Chinese",
        # Tagalog
        "TAGALOG": "Tagalog",
        # Vietnamese
        "VIETNAMESE": "Vietnamese",
        # Korean
        "KOREAN": "Korean",
        # Other
        "OTHER": "Other",
        # Unknown
        "?": "Unknown",
        "-": "Unknown",
        "UNKNOWN": "Unknown",
        "NAN": "Unknown"

    })
)

In [ ]:
master_df["Digital_Engagement"] = (
    master_df["Digital_Engagement"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Digital_Engagement"] = (
    master_df["Digital_Engagement"]
    .replace({

        # High
        "HIGH": "High",
        "HI": "High",

        # Medium
        "MEDIUM": "Medium",
        "MED": "Medium",

        # Low
        "LOW": "Low",

        # No usage
        "NONE": "No Usage",
        "NO USAGE": "No Usage",

        # Unknown
        "NAN": "Unknown",
        "?": "Unknown",
        "-": "Unknown"

    })
)

In [ ]:
master_df["Family_History_Diabetes"] = (
    master_df["Family_History_Diabetes"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Family_History_Diabetes"] = (
    master_df["Family_History_Diabetes"]
    .replace({

        "YES": "Yes",
        "Y": "Yes",
        "TRUE": "Yes",
        "1": "Yes",

        "NO": "No",
        "N": "No",
        "FALSE": "No",
        "0": "No"

    })
)

In [ ]:
master_df["Family_History_Hypertension"] = (
    master_df["Family_History_Hypertension"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Family_History_Hypertension"] = (
    master_df["Family_History_Hypertension"]
    .replace({

        "YES": "Yes",
        "Y": "Yes",
        "TRUE": "Yes",
        "1": "Yes",

        "NO": "No",
        "N": "No",
        "FALSE": "No",
        "0": "No"

    })
)

In [ ]:
master_df["Race_Ethnicity"] = (
    master_df["Race_Ethnicity"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Race_Ethnicity"] = (
    master_df["Race_Ethnicity"]
    .replace({

        # White
        "WHITE": "White",
        "WHITE/CAUCASIAN": "White",
        "CAUCASIAN": "White",

        # Black
        "BLACK": "Black/African American",
        "BLACK/AFRICAN AMERICAN": "Black/African American",
        "AFRICAN AMERICAN": "Black/African American",

        # Asian
        "ASIAN": "Asian",
        "ASIAN AMERICAN": "Asian",

        # Hispanic
        "HISPANIC": "Hispanic/Latino",
        "LATINO": "Hispanic/Latino",
        "HISPANIC/LATINO": "Hispanic/Latino",

        # Pacific Islander
        "NATIVE HAWAIIAN/PACIFIC ISLANDER":
            "Native Hawaiian/Pacific Islander",

        "PACIFIC ISLANDER":
            "Native Hawaiian/Pacific Islander",

        "NHPI":
            "Native Hawaiian/Pacific Islander",

        # Multiracial
        "MULTIRACIAL": "Multiracial/Other",
        "MULTIRACIAL/OTHER": "Multiracial/Other",
        "MIXED": "Multiracial/Other",
        "OTHER": "Multiracial/Other",

        # Unknown
        "?": "Unknown",
        "-": "Unknown",
        "UNKNOWN": "Unknown",
        "NAN": "Unknown"

    })
)

# 4. EXPLORATORY DATA ANALYSIS (EDA)
### Objective

Exploratory Data Analysis was performed to identify patterns, trends, relationships, and anomalies within the member population.

The objective is to understand demographic characteristics, clinical risk factors, healthcare utilization patterns, and healthcare costs before proceeding to feature engineering and predictive modeling.

## PART 1: UNIVARIATE ANALYSIS

In [ ]:
#===========================
# Age Distribution
#===========================

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

sns.histplot(
    master_df["Age"],
    bins=20,
    kde=True
)

plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Member Count")

plt.show()

### Observation

Analyze whether the member population is concentrated within specific age groups.

### Business Interpretation

Understanding age distribution helps identify target populations for preventive care programs and chronic disease management initiatives.

In [ ]:
#=============================
# BMI Distribution
#=============================

plt.figure(figsize=(10,5))

sns.histplot(
    master_df["BMI"],
    bins=25,
    kde=True,
    color="green"
)

plt.title("BMI Distribution")

plt.show()

### Observation

Review whether a significant portion of members fall within overweight or obese BMI ranges.

### Business Interpretation

A higher prevalence of elevated BMI may indicate increased future chronic disease burden.

In [ ]:
#==============================
# Glucose Distribution
#==============================

plt.figure(figsize=(10,5))

sns.histplot(
    master_df["Glucose"],
    bins=25,
    kde=True,
    color="orange"
)

plt.title("Glucose Distribution")

plt.show()

In [ ]:
#====================================
# Healthcare Cost Distribution
#====================================

plt.figure(figsize=(10,5))

sns.histplot(
    master_df["Costs"],
    bins=30,
    kde=True,
    color="red"
)

plt.title("Healthcare Cost Distribution")

plt.show()

Healthcare cost distributions are often right-skewed, with a small population of members accounting for a large share of healthcare expenditure.

## PART 2: BIVARIATE ANALYSIS

In [ ]:
#========================
# BMI vs GLucose
#========================
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=master_df,
    x="BMI",
    y="Glucose"
)

plt.title("BMI vs Glucose")

plt.show()

A positive relationship between BMI and glucose levels may indicate increased diabetes risk among overweight members.

In [ ]:
#==============================
# Exercise Frequency vs Costs
#==============================

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=master_df,
    x="Exercise_Frequency",
    y="Costs"
)

plt.title("Exercise Frequency vs Healthcare Costs")
plt.show()

In [ ]:
#==============================
# Smoking Status vs Costs
#==============================
# Identify cost burden of smoking.

plt.figure(figsize=(8,6))
sns.boxplot(
    data=master_df,
    x="Smoking_Status",
    y="Costs"
)

plt.xticks(rotation=45)
plt.show()

In [ ]:
master_df["Screening_Participation"] = (
    master_df["Screening_Participation"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_df["Screening_Participation"] = (
    master_df["Screening_Participation"]
    .replace({

        "YES": "Yes",
        "Y": "Yes",
        "TRUE": "Yes",
        "1": "Yes",

        "NO": "No",
        "N": "No",
        "FALSE": "No",
        "0": "No",

        "?": "Unknown",
        "-": "Unknown",
        "UNKNOWN": "Unknown"

    })
)

In [ ]:
#=================================
# Screening Participation vs Costs
#=================================
# Validate preventive care effectiveness

plt.figure(figsize=(8,6))

sns.boxplot(
    data=master_df,
    x="Screening_Participation",
    y="Costs"
)

plt.show()

## PART 3: OUTLIER ANALYSIS

In [ ]:
#=======================
# BMI outliers
#=======================
plt.figure(figsize=(10,4))
sns.boxplot(x=master_df["BMI"])
plt.title("BMI Outlier Analysis")
plt.show()

In [ ]:
#=========================
# Glucose outliers
#=========================
plt.figure(figsize=(10,4))
sns.boxplot(x=master_df["Glucose"])
plt.title("Glucose Outlier Analysis")
plt.show()

In [ ]:
#==================
# Cost Outliers
#==================

plt.figure(figsize=(10,4))
sns.boxplot(x=master_df["Costs"])
plt.title("Healthcare Cost Outlier Analysis")
plt.show()

In [ ]:
#==========================
# Visit outliers
#==========================
plt.figure(figsize=(10,4))
sns.boxplot(x=master_df["Visits"])
plt.title("Visits Outlier Analysis")
plt.show()

# EDA Key Findings

1. The member population spans a broad age range, indicating diverse healthcare needs across demographic groups.

2. BMI distribution suggests a substantial proportion of members fall within overweight and obese categories, potentially increasing future chronic disease burden.

3. Glucose levels exhibit considerable variation across the member population, indicating the presence of normal, prediabetic, and diabetic risk profiles.

4. Healthcare costs are heavily right-skewed, suggesting a relatively small group of members account for a large proportion of total healthcare expenditure.

5. A positive relationship is observed between BMI and glucose levels, supporting the association between obesity and diabetes risk.

6. Members with higher exercise frequency generally demonstrate lower healthcare cost patterns.

7. Preventive screening participants exhibit different healthcare cost patterns compared to non-participants, highlighting the importance of preventive care.

8. Smoking status appears associated with healthcare expenditure differences, indicating lifestyle factors influence healthcare utilization.

9. Clinical indicators such as BMI, glucose, blood pressure, and HbA1c demonstrate substantial variability and are expected to be strong predictors of future chronic disease.

10. Several healthcare variables contain notable outliers, particularly healthcare costs and utilization metrics, suggesting the presence of a high-risk, high-cost member segment.

# 5. FEATURE ENGINEERING

Creating meaningful variables from existing data, instead of using raw columns.
Raw data ususally does not perform as well as engineered fetures.
- Better ML performance
- Better Business understanding

Feature engineering was performed to create healthcare-specific indicators that improve analytical insights, support predictive modeling, enhance population segmentation, and simplify business interpretation.

The engineered features combine demographic, lifestyle, clinical, preventive care, and healthcare utilization data into meaningful healthcare metrics.

In [ ]:
# =====================================================
# Create Working Copy for Feature Engineering
# =====================================================

feature_df = master_df.copy()

print("Feature engineering dataset created successfully.")

In [ ]:
feature_df["Age"].dtype

In [ ]:
feature_df["Region"].value_counts(dropna=False).head(50)

In [ ]:
# =====================================================
# Region Standardization
# =====================================================

feature_df["Region"] = (

    feature_df["Region"]
    .astype(str)
    .str.strip()
    .str.upper()

)

region_mapping = {

    # ---------------------------------
    # NORTHERN CALIFORNIA
    # ---------------------------------

    "NORTHERN CALIFORNIA":
        "Northern California",

    "NORTHERN CAL":
        "Northern California",

    "N. CALIFORNIA":
        "Northern California",

    "NORCAL":
        "Northern California",

    # ---------------------------------
    # SOUTHERN CALIFORNIA
    # ---------------------------------

    "SOUTHERN CALIFORNIA":
        "Southern California",

    "SOUTHERN CAL":
        "Southern California",

    "S. CALIFORNIA":
        "Southern California",

    "SOCAL":
        "Southern California",

    # ---------------------------------
    # MID-ATLANTIC STATES
    # ---------------------------------

    "MID-ATLANTIC STATES":
        "Mid-Atlantic States",

    "MID ATLANTIC":
        "Mid-Atlantic States",

    "MID-ATLANTIC":
        "Mid-Atlantic States",

    "MAS":
        "Mid-Atlantic States",

    # ---------------------------------
    # NORTHWEST
    # ---------------------------------

    "NORTHWEST":
        "Northwest",

    "NORTH WEST":
        "Northwest",

    "NW":
        "Northwest",

    # ---------------------------------
    # WASHINGTON
    # ---------------------------------

    "WASHINGTON":
        "Washington",

    "WASHINGTON STATE":
        "Washington",

    "WA":
        "Washington",

    # ---------------------------------
    # GEORGIA
    # ---------------------------------

    "GEORGIA":
        "Georgia",

    "GA":
        "Georgia",

    # ---------------------------------
    # COLORADO
    # ---------------------------------

    "COLORADO":
        "Colorado",

    "CO":
        "Colorado",

    # ---------------------------------
    # HAWAII
    # ---------------------------------

    "HAWAII":
        "Hawaii",

    "HAWAI'I":
        "Hawaii",

    "HI":
        "Hawaii",

    # ---------------------------------
    # NEVADA
    # ---------------------------------

    "NEVADA":
        "Nevada",

    "NV":
        "Nevada"

}

feature_df["Region"] = (
    feature_df["Region"]
    .replace(region_mapping)
)

In [ ]:
feature_df["Region"].value_counts(
    dropna=False
)

In [ ]:
# =====================================================
# Existing Hypertension Flag Standardization
# =====================================================

feature_df["Existing_Hypertension_Flag"] = (
    feature_df["Existing_Hypertension_Flag"]
    .astype(str)
    .str.strip()
    .str.upper()
)

hypertension_mapping = {

    "Y": "Yes",
    "YES": "Yes",
    "1": "Yes",
    "TRUE": "Yes",

    "N": "No",
    "NO": "No",
    "0": "No",
    "FALSE": "No"

}

feature_df["Existing_Hypertension_Flag"] = (
    feature_df["Existing_Hypertension_Flag"]
    .replace(hypertension_mapping)
)

In [ ]:
# =====================================================
# Clean Age Column Properly
# =====================================================

# Extract numeric portion if text exists
feature_df["Age"] = (
    feature_df["Age"]
    .astype(str)
    .str.extract(r'(\d+)')[0]
)

# Convert to numeric
feature_df["Age"] = pd.to_numeric(
    feature_df["Age"],
    errors="coerce"
)

print(feature_df["Age"].dtype)

feature_df["Age"].describe()

In [ ]:
# =====================================================
# Feature 1: Age Group
# =====================================================

feature_df["Age_Group"] = pd.cut(
    feature_df["Age"],
    bins=[0, 30, 45, 60, 100],
    labels=[
        "18-30",
        "31-45",
        "46-60",
        "60+"
    ]
)

feature_df["Age_Group"].value_counts()

In [ ]:
feature_df[["Age", "Age_Group"]].head()

In [ ]:
# =====================================================
# Feature 2: Wellness Score
# =====================================================

feature_df["Screening_Score"] = np.where(
    feature_df["Screening_Participation"] == "Yes",
    100,
    0
)

feature_df["Wellness_Score"] = (
    feature_df["Exercise_Frequency"].fillna(0) * 10 +
    feature_df["Diet_Score"].fillna(0) +
    feature_df["Screening_Score"]

) / 3

feature_df["Wellness_Score"].describe()

In [ ]:
# =====================================================
# Feature 3: Lifestyle Risk Score
# =====================================================

feature_df["Smoking_Risk"] = np.where(
    feature_df["Smoking_Status"] == "Smoker",
    100,
    0
)

feature_df["BMI_Risk"] = np.where(
    feature_df["BMI"] >= 30, 100,
    np.where(feature_df["BMI"] >= 25, 50,0)
)

feature_df["Exercise_Risk"] = np.where(
    feature_df["Exercise_Frequency"] < 2, 100,
    np.where(feature_df["Exercise_Frequency"] < 4, 50, 0)
)

feature_df["Lifestyle_Risk_Score"] = (
    feature_df["Smoking_Risk"] +
    feature_df["BMI_Risk"] +
    feature_df["Exercise_Risk"]

) / 3

feature_df["Lifestyle_Risk_Score"].describe()

In [ ]:
# =====================================================
# Feature 4: Preventive Compliance Score
# =====================================================

feature_df["Screening_Points"] = np.where(
    feature_df["Screening_Participation"] == "Yes", 100, 0
)

feature_df["Preventive_Compliance_Score"] = (
    feature_df["Screening_Points"] +
    feature_df["Preventive_Screenings_Completed"].fillna(0) * 10 +
    feature_df["Health_Risk_Assessment_Completed"].fillna(0) * 20
) / 3

feature_df["Preventive_Compliance_Score"].describe()

In [ ]:
# =====================================================
# Feature 5: Chronic Disease Risk Index
# =====================================================

feature_df["BMI_Norm"] = (feature_df["BMI"] / feature_df["BMI"].max())

feature_df["Glucose_Norm"] = (feature_df["Glucose"] / feature_df["Glucose"].max())

feature_df["HbA1c_Norm"] = (feature_df["HbA1c"] / feature_df["HbA1c"].max())

feature_df["BP_Norm"] = (feature_df["Systolic_BP"] / feature_df["Systolic_BP"].max())

feature_df["Cholesterol_Norm"] = (feature_df["Cholesterol"] / feature_df["Cholesterol"].max())

feature_df["Chronic_Disease_Risk_Index"] = (
    feature_df["BMI_Norm"] +
    feature_df["Glucose_Norm"] +
    feature_df["HbA1c_Norm"] +
    feature_df["BP_Norm"] +
    feature_df["Cholesterol_Norm"]
) / 5

feature_df["Chronic_Disease_Risk_Index"].describe()

In [ ]:
# =====================================================
# Feature 6: Cost Per Visit
# =====================================================

feature_df["Cost_Per_Visit"] = np.where(
    feature_df["Visits"] > 0,
    feature_df["Costs"] / feature_df["Visits"],0
)

feature_df["Cost_Per_Visit"].describe()

In [ ]:
engineered_features = [
    "Age_Group",
    "Wellness_Score",
    "Lifestyle_Risk_Score",
    "Preventive_Compliance_Score",
    "Chronic_Disease_Risk_Index",
    "Cost_Per_Visit"
]

feature_df[engineered_features].head()

## Feature Engineering Summary

The following engineered features were created:

1. Age_Group
2. Wellness_Score
3. Lifestyle_Risk_Score
4. Preventive_Compliance_Score
5. Chronic_Disease_Risk_Index
6. Cost_Per_Visit

These features capture key healthcare concepts including demographic segmentation, healthy lifestyle behavior, preventive-care engagement, chronic disease risk, and healthcare utilization efficiency.

The engineered features will be used in subsequent phases including machine learning, clustering, and Power BI reporting.

In [ ]:
feature_df[
    [
        "Age_Group",
        "Wellness_Score",
        "Lifestyle_Risk_Score",
        "Preventive_Compliance_Score",
        "Chronic_Disease_Risk_Index",
        "Cost_Per_Visit"
    ]
].describe(include="all")

In [ ]:
numeric_cols = [

    "Age",
    "BMI",
    "Glucose",
    "HbA1c",
    "Cholesterol",
    "LDL_Cholesterol",
    "HDL_Cholesterol",
    "Triglycerides",
    "Systolic_BP",
    "Diastolic_BP",
    "Exercise_Frequency",
    "Diet_Score",
    "Visits",
    "ER_Visits",
    "Hospital_Admissions",
    "Medications",
    "Costs",
    "Chronic_Condition_Count"

]

feature_df[numeric_cols].describe().T

In [ ]:
impute_cols = [

    "Age",
    "BMI",
    "Glucose",
    "HbA1c",
    "Cholesterol",
    "LDL_Cholesterol",
    "HDL_Cholesterol",
    "Triglycerides",
    "Systolic_BP",
    "Diastolic_BP",
    "Exercise_Frequency",
    "Diet_Score",
    "Visits",
    "ER_Visits",
    "Hospital_Admissions",
    "Medications",
    "Costs",
    "Chronic_Condition_Count"

]

for col in impute_cols:

    feature_df[col] = feature_df[col].fillna(
        feature_df[col].median()
    )

In [ ]:
# =====================================================
# Feature 7: Blood Pressure Category
# =====================================================

feature_df["BP_Category"] = np.select(
    [
        feature_df["Systolic_BP"] < 120,
        ((feature_df["Systolic_BP"] >= 120) & (feature_df["Systolic_BP"] < 130)),
        ((feature_df["Systolic_BP"] >= 130) & (feature_df["Systolic_BP"] < 140)),
        feature_df["Systolic_BP"] >= 140
    ],

    [
        "Normal",
        "Elevated",
        "Stage 1 Hypertension",
        "Stage 2 Hypertension"
    ],
    default="Unknown"
)
feature_df["BP_Category"].value_counts()

In [ ]:
# =====================================================
# Feature 8: Family Risk Score
# =====================================================

feature_df["Family_Risk_Score"] = (
    np.where(
        feature_df["Family_History_Diabetes"] == "Yes", 1, 0
    )
    +
    np.where(
        feature_df["Family_History_Hypertension"] == "Yes", 1, 0
    )
)
feature_df["Family_Risk_Score"].value_counts()

In [ ]:
# =====================================================
# Feature 9: Utilization Risk Score
# =====================================================

feature_df["Utilization_Risk_Score"] = (

    feature_df["Visits"].fillna(0) +
    (feature_df["ER_Visits"].fillna(0) * 2) +
    (feature_df["Hospital_Admissions"].fillna(0) * 3) +
    feature_df["Medications"].fillna(0)
)

feature_df["Utilization_Risk_Score"].describe()

In [ ]:
# =====================================================
# Feature 10: Diabetes Risk Flag
# =====================================================

feature_df["Diabetes_Risk_Flag"] = np.select(
    [
        ((feature_df["Glucose"] >= 126) | (feature_df["HbA1c"] >= 6.5)),

        ((feature_df["Glucose"] >= 100) | (feature_df["HbA1c"] >= 5.7))
    ],

    ["High Risk", "Moderate Risk"],
    default="Low Risk"
)
feature_df["Diabetes_Risk_Flag"].value_counts()

In [ ]:
# =====================================================
# Feature 11: Wellness Engagement Category
# =====================================================

feature_df["Wellness_Engagement_Category"] = pd.cut(
    feature_df["Wellness_Score"],
    bins=[-1, 33, 66, 100],
    labels=[
        "Low Engagement",
        "Moderate Engagement",
        "High Engagement"
    ]
)
feature_df["Wellness_Engagement_Category"].value_counts()

In [ ]:
advanced_features = [
    "BP_Category",
    "Family_Risk_Score",
    "Utilization_Risk_Score",
    "Diabetes_Risk_Flag",
    "Wellness_Engagement_Category"
]
feature_df[advanced_features].head()

## Feature Engineering Summary

The following healthcare-oriented features were engineered:

### Mandatory Features

1. Age_Group
2. Wellness_Score
3. Lifestyle_Risk_Score
4. Preventive_Compliance_Score
5. Chronic_Disease_Risk_Index
6. Cost_Per_Visit

### Advanced Healthcare Features

7. BP_Category
8. Family_Risk_Score
9. Utilization_Risk_Score
10. Diabetes_Risk_Flag
11. Wellness_Engagement_Category

These engineered variables improve disease risk assessment, preventive care evaluation, healthcare utilization monitoring, population segmentation, and predictive modeling performance.

The resulting dataset is now suitable for machine learning model development and clustering analysis.

In [ ]:
feature_df["Develops_Chronic_Disease_12M"].value_counts(dropna=False)

feature_df["Develops_Chronic_Disease_12M"].value_counts(normalize=True) * 100

In [ ]:
feature_df["Develops_Chronic_Disease_12M"].isnull().sum()

In [ ]:
feature_df = feature_df.dropna(subset=["Develops_Chronic_Disease_12M"])

In [ ]:
leakage_features = [
    "Develops_Diabetes_12M",
    "Develops_Hypertension_12M"
]

print(leakage_features)

In [ ]:
feature_df["Develops_Chronic_Disease_12M"].value_counts()

In [ ]:
feature_df["Develops_Chronic_Disease_12M"].value_counts(normalize=True)

In [ ]:
feature_df["Region"].value_counts().tail(20)

In [ ]:
feature_df["Age_Group"] = (
    feature_df["Age_Group"]
    .astype(str)
    .replace("nan", "Unknown")
)

In [ ]:
feature_df["Cost_Per_Visit"] = (
    feature_df["Cost_Per_Visit"]
    .fillna(0)
)

In [ ]:
feature_df["Chronic_Disease_Risk_Index"] = (
    feature_df["Chronic_Disease_Risk_Index"]
    .fillna(
        feature_df["Chronic_Disease_Risk_Index"].median()
    )
)

In [ ]:
feature_df[
    [
        "Chronic_Disease_Risk_Index",
        "Cost_Per_Visit",
        "Age_Group"
    ]
].isnull().sum()

In [ ]:
# =====================================================
# Create ML Working Dataset
# =====================================================

ml_df = feature_df.copy()

print("Shape of ML Dataset:")
print(ml_df.shape)

In [ ]:
# to prevent data leakage these columns have been dropped

cols_to_drop = [
    "Member_ID",
    "Develops_Diabetes_12M",
    "Develops_Hypertension_12M"
]

ml_df = ml_df.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
ml_df.to_csv("ML_Ready_Dataset.csv", index=False)
print("ML Ready Dataset Created Successfully.")

## Feature Selection Strategy

The following groups of variables were selected:

### Demographic Features

Age
Gender
Age_Group
Region

### Lifestyle Features

Exercise_Frequency
Smoking_Status
Diet_Score

### Clinical Features

BMI
Glucose
HbA1c
Cholesterol
LDL_Cholesterol
HDL_Cholesterol
Triglycerides
Systolic_BP
Diastolic_BP

### Wellness Features

Wellness_Score
Preventive_Compliance_Score

### Claims Features

Visits
ER_Visits
Hospital_Admissions
Costs
Cost_Per_Visit
Utilization_Risk_Score

### Family History

Family_History_Diabetes
Family_History_Hypertension
Family_Risk_Score

### Engineered Features

Lifestyle_Risk_Score
Chronic_Disease_Risk_Index
Diabetes_Risk_Flag

In [ ]:
# =====================================================
# Define Target Variable
# =====================================================

TARGET = "Develops_Chronic_Disease_12M"

In [ ]:
# =====================================================
# Features for Modeling
# =====================================================

selected_features = [
    # Demographics
    "Age",
    "Gender",
    "Region",
    "Age_Group",

    # Lifestyle
    "Exercise_Frequency",
    "Smoking_Status",
    "Diet_Score",

    # Clinical
    "BMI",
    "Glucose",
    "HbA1c",
    "Cholesterol",
    "LDL_Cholesterol",
    "HDL_Cholesterol",
    "Triglycerides",
    "Systolic_BP",
    "Diastolic_BP",

    # Wellness
    "Wellness_Score",
    "Preventive_Compliance_Score",

    # Claims
    "Visits",
    "ER_Visits",
    "Hospital_Admissions",
    "Medications",
    "Costs",
    "Cost_Per_Visit",

    # Family Risk
    "Family_History_Diabetes",
    "Family_History_Hypertension",
    "Family_Risk_Score",

    # Engineered Features
    "Lifestyle_Risk_Score",
    "Chronic_Disease_Risk_Index",
    "Utilization_Risk_Score",

    # Categories
    "BP_Category",
    "Diabetes_Risk_Flag",
    "Wellness_Engagement_Category"
]

In [ ]:
# =====================================================
# Create Modeling Dataset
# =====================================================

model_df = ml_df[selected_features + [TARGET]].copy()
print(model_df.shape)
model_df.head()

In [ ]:
model_df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# =====================================================
# Identify Numerical and Categorical Columns
# =====================================================

numeric_features = model_df.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = model_df.select_dtypes(include=["object", "category"]).columns.tolist()

if TARGET in numeric_features:
    numeric_features.remove(TARGET)

if TARGET in categorical_features:
    categorical_features.remove(TARGET)

print("Numeric Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

In [ ]:
missing_summary = pd.DataFrame({
    "Feature":model_df.columns,
    "Missing_Count": model_df.isnull().sum().values,
    "Missing_Percent":(model_df.isnull().mean()*100).round(2).values
})

missing_summary.sort_values("Missing_Percent", ascending=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# =====================================================
# Target Class Distribution
# =====================================================

plt.figure(figsize=(7,5))
sns.countplot(x=TARGET, data=model_df)
plt.title("Target Variable Distribution")
plt.xlabel("Develops Chronic Disease Within 12 Months")
plt.ylabel("Member Count")
plt.show()

In [ ]:
# =====================================================
# Exact Duplicate Records
# =====================================================

duplicate_rows = feature_df.duplicated().sum()

print("Duplicate Rows:", duplicate_rows)

In [ ]:
feature_df["Member_ID"].nunique()

In [ ]:
print("Total Rows:")
print(feature_df.shape[0])

print("\nUnique Members:")
print(feature_df["Member_ID"].nunique())

print("\nDuplicate Member IDs:")
print(feature_df["Member_ID"].duplicated().sum())

In [ ]:
feature_df.groupby(
    "Member_ID"
)[
    "Develops_Chronic_Disease_12M"
].nunique().value_counts()

In [ ]:
feature_df.groupby("Member_ID").size().describe()

In [ ]:
conflicting_members = (

    feature_df.groupby("Member_ID")[
        "Develops_Chronic_Disease_12M"
    ]
    .nunique()
)

conflicting_members = conflicting_members[
    conflicting_members > 1
]

print(conflicting_members.shape)

In [ ]:
sample_members = conflicting_members.index[:5]

feature_df[
    feature_df["Member_ID"].isin(sample_members)
][

    [
        "Member_ID",
        "Develops_Chronic_Disease_12M",
        "Develops_Diabetes_12M",
        "Develops_Hypertension_12M"
    ]

].sort_values("Member_ID")

In [ ]:
[c for c in feature_df.columns if "date" in c.lower()]

In [ ]:
[c for c in feature_df.columns if "year" in c.lower()]

In [ ]:
date_cols = [

    "Member_Since_Date",
    "Record_Date",
    "Screening_Date",
    "Period_End_Date"

]

for col in date_cols:

    print("\n", col)
    print(feature_df[col].dtype)

In [ ]:
# =====================================================
# Convert Date Columns
# =====================================================

date_columns = [
    "Member_Since_Date",
    "Record_Date",
    "Screening_Date",
    "Period_End_Date"
]

for col in date_columns:
    if col in feature_df.columns:
        feature_df[col] = pd.to_datetime(
            feature_df[col],
            errors="coerce"
        )

print("Date conversion completed.")

In [ ]:
feature_df[
    [
        "Record_Date",
        "Screening_Date",
        "Period_End_Date"
    ]
].isnull().sum()

In [ ]:
feature_df["SDOH_Vulnerability_Index"] = pd.to_numeric(feature_df["SDOH_Vulnerability_Index"],errors="coerce")
print(feature_df["SDOH_Vulnerability_Index"].dtype)

In [ ]:
feature_df["Tenure_Years"] = pd.to_numeric(feature_df["Tenure_Years"],errors="coerce")
print(feature_df["Tenure_Years"].dtype)

In [ ]:
latest_member_df = (feature_df.sort_values("Screening_Date").groupby("Member_ID").tail(1))

In [ ]:
latest_member_df.shape

In [ ]:
# =====================================================
# Create Snapshot Date
# =====================================================

feature_df["Snapshot_Date"] = feature_df[
    [
        "Screening_Date",
        "Record_Date",
        "Period_End_Date"
    ]
].max(axis=1)

print("Snapshot Date Created")

In [ ]:
# =====================================================
# Sort Dataset By Latest Record
# =====================================================

feature_df = feature_df.sort_values("Snapshot_Date")

In [ ]:
# =====================================================
# Create Member-Level Dataset
# =====================================================

member_level_df = (feature_df.groupby("Member_ID").tail(1).reset_index(drop=True))

print("Rows:", member_level_df.shape[0])
print("Unique Members:", member_level_df["Member_ID"].nunique())

In [ ]:
member_level_df["Develops_Chronic_Disease_12M"].value_counts()

In [ ]:
member_level_df["Develops_Chronic_Disease_12M"].value_counts(normalize=True)

In [ ]:
member_level_df.shape

In [ ]:
member_level_df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
member_level_df.drop(columns=["Health_Risk_Assessment_Completed"],inplace=True)

In [ ]:
ml_df = member_level_df.copy()

In [ ]:
cols_to_drop = [

    # IDs
    "Member_ID",

    # Future outcomes (leakage)
    "Develops_Diabetes_12M",
    "Develops_Hypertension_12M",

    # Dates
    "Member_Since_Date",
    "Screening_Date",
    "Record_Date",
    "Period_End_Date",
    "Snapshot_Date",

    # Helper engineering columns
    "BMI_Norm",
    "Glucose_Norm",
    "HbA1c_Norm",
    "BP_Norm",
    "Cholesterol_Norm",

    # Replaced variables
    "Blood_Pressure",

    # Fully missing
    "Health_Risk_Assessment_Completed"

]

ml_df = ml_df.drop(columns=cols_to_drop,errors="ignore")

print("Shape after cleanup:")
print(ml_df.shape)

In [ ]:
missing_check = (ml_df.isnull().sum().sort_values(ascending=False))

missing_check[missing_check > 0]

In [ ]:
# =====================================================
# Final Missing Value Imputation
# =====================================================

# Impute SDOH Vulnerability Index
feature_df["SDOH_Vulnerability_Index"] = (feature_df["SDOH_Vulnerability_Index"].fillna(feature_df["SDOH_Vulnerability_Index"].median()))

# Impute Tenure Years
feature_df["Tenure_Years"] = (feature_df["Tenure_Years"].fillna(feature_df["Tenure_Years"].median()))

print("Missing value imputation completed.")

In [ ]:
# =====================================================
# Create Final ML Dataset
# =====================================================

ml_df = member_level_df.copy()
# Target Variable
TARGET = "Develops_Chronic_Disease_12M"

# Columns to Remove
cols_to_drop = [

    # IDs
    "Member_ID",
    "Region_ID",
    "Home_Facility_ID",
    "Wellness_Record_ID",
    "Program_ID",
    "Screening_ID",
    "Facility_ID",
    "Claim_Record_ID",

    # Future Outcome Leakage
    "Develops_Diabetes_12M",
    "Develops_Hypertension_12M",

    # Date Fields
    "Member_Since_Date",
    "Record_Date",
    "Screening_Date",
    "Period_End_Date",
    "Snapshot_Date",

    # Helper Feature Columns
    "BMI_Norm",
    "Glucose_Norm",
    "HbA1c_Norm",
    "BP_Norm",
    "Cholesterol_Norm",

    # Replaced Feature
    "Blood_Pressure"

]

ml_df = ml_df.drop(columns=cols_to_drop, errors="ignore")

print("ML Dataset Shape:")
print(ml_df.shape)

In [ ]:
# =====================================================
# Final ML Imputation
# =====================================================

ml_df["SDOH_Vulnerability_Index"] = (pd.to_numeric(ml_df["SDOH_Vulnerability_Index"],errors="coerce"))

ml_df["Tenure_Years"] = (pd.to_numeric(ml_df["Tenure_Years"],errors="coerce"))

ml_df["SDOH_Vulnerability_Index"] = (ml_df["SDOH_Vulnerability_Index"].fillna(ml_df["SDOH_Vulnerability_Index"].median()))

ml_df["Tenure_Years"] = (ml_df["Tenure_Years"].fillna(ml_df["Tenure_Years"].median()))

In [ ]:
ml_df[["SDOH_Vulnerability_Index", "Tenure_Years"]].isnull().sum()

In [ ]:
# =====================================================
# Identify Numeric and Categorical Features
# =====================================================

numeric_features = ml_df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = ml_df.select_dtypes(include=["object", "category"]).columns.tolist()

if TARGET in numeric_features:
    numeric_features.remove(TARGET)

if TARGET in categorical_features:
    categorical_features.remove(TARGET)

print("Numeric Features:", len(numeric_features))
print("Categorical Features:", len(categorical_features))

In [ ]:
# =====================================================
# Final Null Value Validation
# =====================================================

missing_summary = pd.DataFrame({
    "Missing_Count":ml_df.isnull().sum(),
    "Missing_Percent":(ml_df.isnull().mean() * 100).round(2)
})

missing_summary = missing_summary[missing_summary["Missing_Count"] > 0]
missing_summary.sort_values("Missing_Count", ascending=False)

In [ ]:
# =====================================================
# One-Hot Encoding
# =====================================================

model_encoded = pd.get_dummies(ml_df, columns=categorical_features, drop_first=True)

print("Encoded Dataset Shape:")
print(model_encoded.shape)

In [ ]:
# =====================================================
# Create Feature Matrix and Target Vector
# =====================================================

X = model_encoded.drop(TARGET, axis=1)
y = model_encoded[TARGET]

print("Feature Matrix:", X.shape)
print("Target Vector:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# =====================================================
# Train-Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print("Training Set:", X_train.shape)
print("Testing Set :", X_test.shape)

In [ ]:
# =====================================================
# Training Target Distribution
# =====================================================

class_distribution = pd.DataFrame({
    "Count": y_train.value_counts(),
    "Percentage":(y_train.value_counts(normalize=True) * 100).round(2)
})

class_distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================
# Visualize Class Distribution
# =====================================================

plt.figure(figsize=(6,4))
sns.countplot(x = y_train)
plt.title("Target Class Distribution")
plt.xlabel("Develops Chronic Disease (12 Months)")
plt.ylabel("Member Count")
plt.show()

## Class Imbalance Handling
The target variable exhibits class imbalance, with approximately 10% of members developing chronic disease within 12 months.

To improve the model's ability to identify high-risk members, SMOTE (Synthetic Minority Oversampling Technique) is applied to the training data only.

This helps ensure that disease cases receive sufficient representation during model training.

In [ ]:
# =====================================================
# Import SMOTE
# =====================================================

from imblearn.over_sampling import SMOTE

# =====================================================
# Apply SMOTE ONLY on Training Data
# =====================================================

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Original Training Shape:")
print(X_train.shape)

print("\nSMOTE Training Shape:")
print(X_train_smote.shape)

In [ ]:
# =====================================================
# Verify Balanced Classes
# =====================================================

balanced_distribution = pd.DataFrame({
    "Count": y_train_smote.value_counts(),
    "Percentage":(y_train_smote.value_counts(normalize=True)* 100).round(2)
})

balanced_distribution

In [ ]:
# =====================================================
# Class Distribution After SMOTE
# =====================================================

plt.figure(figsize=(6,4))
sns.countplot(x=y_train_smote)
plt.title("Balanced Training Data After SMOTE")
plt.xlabel("Develops Chronic Disease")
plt.ylabel("Member Count")
plt.show()

In [ ]:
# =====================================================
# Machine Learning Evaluation Libraries
# =====================================================

import numpy as np
import pandas as pd

from itertools import product

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,

    confusion_matrix,
    classification_report,

    roc_curve,
    precision_recall_curve

)

from sklearn.model_selection import (

    StratifiedKFold,
    cross_val_score

)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
# =====================================================
# Generate All Hyperparameter Combinations
# =====================================================

def create_param_grid(param_dict):
    
    keys = list(param_dict.keys())
    
    values = list(param_dict.values())
    
    combinations = [
        
        dict(zip(keys, combo))
        
        for combo in product(*values)
        
    ]
    
    return combinations

In [ ]:
# =====================================================
# Cross Validation
# =====================================================

def cross_validate_model(
    model,
    X_train,
    y_train
):
    
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    
    cv_scores = cross_val_score(
        
        model,
        
        X_train,
        y_train,
        
        cv=skf,
        
        scoring="recall"
        
    )
    
    return cv_scores.mean()

In [ ]:
# =====================================================
# Generic Experiment Runner
# =====================================================

def run_hyperparameter_experiment(

    model_class,

    parameter_grid,

    model_name,

    X_train,
    y_train,

    X_test,
    y_test

):
    
    results = []
    
    for params in parameter_grid:
        
        model = model_class(**params)
        
        model.fit(
            X_train,
            y_train
        )
        
        y_pred = model.predict(X_test)
        
        y_prob = model.predict_proba(
            X_test
        )[:,1]
        
        cv_recall = np.nan
        
        results.append({
            
            "Model": model_name,
            
            **params,
            
            "Accuracy":
                accuracy_score(
                    y_test,
                    y_pred
                ),
            
            "Precision":
                precision_score(
                    y_test,
                    y_pred
                ),
            
            "Recall":
                recall_score(
                    y_test,
                    y_pred
                ),
            
            "F1":
                f1_score(
                    y_test,
                    y_pred
                ),
            
            "ROC_AUC":
                roc_auc_score(
                    y_test,
                    y_prob
                ),
            
            "CV_Recall":
                cv_recall
            
        })
    
    return pd.DataFrame(results)

In [ ]:
# =====================================================
# Store Experiments
# =====================================================

experiment_results = {}

# 1. LOGISTIC REGRESSION 
The impact of regularization strength on chronic disease prediction performance was evaluated.

Recall remains the primary metric because healthcare organizations prioritize identifying high-risk members.

In [ ]:
from sklearn.linear_model import LogisticRegression

log_params = {

    "C":[
        0.01,
        0.1,
        1,
        10
    ],

    "penalty":[
        "l2"
    ],

    "solver":[
        "liblinear"
    ],

    "class_weight":[
        "balanced"
    ],

    "random_state":[42]

}

log_grid = create_param_grid(
    log_params
)

log_experiment_results = run_hyperparameter_experiment(

    LogisticRegression,

    log_grid,

    "Logistic Regression",

    X_train_smote,
    y_train_smote,

    X_test,
    y_test

)

experiment_results[
    "Logistic Regression"
] = log_experiment_results

display(
    log_experiment_results
    .sort_values(
        "Recall",
        ascending=False
    )
)

# 2. DECISION TREE 

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_params = {

    "max_depth":[
        3,
        5,
        10,
        15
    ],

    "criterion":[
        "gini",
        "entropy"
    ],

    "min_samples_split":[
        2,
        5,
        10
    ],

    "class_weight":[
        "balanced"
    ],

    "random_state":[42]

}

tree_grid = create_param_grid(
    tree_params
)

tree_experiment_results = run_hyperparameter_experiment(

    DecisionTreeClassifier,

    tree_grid,

    "Decision Tree",

    X_train_smote,
    y_train_smote,

    X_test,
    y_test

)

experiment_results[
    "Decision Tree"
] = tree_experiment_results

display(
    tree_experiment_results
    .sort_values(
        "Recall",
        ascending=False
    )
)

# 3. RANDOM FOREST

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {

    "n_estimators":[
        100,
        200
    ],

    "max_depth":[
        5,
        10,
        15
    ],

    "max_features":[
        "sqrt",
        "log2"
    ],

    "class_weight":[
        "balanced"
    ],

    "random_state":[42],

    "n_jobs":[-1]

}

rf_grid = create_param_grid(
    rf_params
)

rf_experiment_results = run_hyperparameter_experiment(

    RandomForestClassifier,

    rf_grid,

    "Random Forest",

    X_train_smote,
    y_train_smote,

    X_test,
    y_test

)

experiment_results[
    "Random Forest"
] = rf_experiment_results

display(
    rf_experiment_results
    .sort_values(
        "Recall",
        ascending=False
    )
)

# 4. GRADIENT BOOSTING

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_params = {

    "n_estimators":[
        # 100,
        200
    ],

    "learning_rate":[
        0.01,
        0.1
    ],

    "max_depth":[
        3,
        5
    ],

    "random_state":[42]

}

gb_grid = create_param_grid(
    gb_params
)

gb_experiment_results = run_hyperparameter_experiment(

    GradientBoostingClassifier,

    gb_grid,
    "Gradient Boosting",
    X_train_smote,
    y_train_smote,
    X_test,
    y_test

)

experiment_results[
    "Gradient Boosting"
] = gb_experiment_results

display(
    gb_experiment_results
    .sort_values(
        "Recall",
        ascending=False
    )
)

In [ ]:
best_models = []

for model_name, df in experiment_results.items():
    
    best_row = (
        
        df
        .sort_values(
            ["Recall", "ROC_AUC"],
            ascending=False
        )
        .head(1)
        
    )
    
    best_models.append(best_row)

best_models_df = pd.concat(
    best_models,
    ignore_index=True
)

display(best_models_df.round(4))

In [ ]:
final_model_comparison = (

    best_models_df[

        [
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC_AUC",
            "CV_Recall"
        ]

    ]

    .sort_values(
        "Recall",
        ascending=False
    )

)

display(
    final_model_comparison.round(4)
)

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    data=final_model_comparison,

    x="Model",

    y="Recall",

    hue="Model",

    legend=False

)

plt.title(
    "Final Recall Comparison"
)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    data=final_model_comparison,

    x="Model",

    y="ROC_AUC",

    hue="Model",

    legend=False

)

plt.title(
    "Final ROC-AUC Comparison"
)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    data=final_model_comparison,

    x="Model",

    y="F1",

    hue="Model",

    legend=False

)

plt.title(
    "Final F1 Comparison"
)

plt.show()

In [ ]:
metrics_plot = final_model_comparison.melt(

    id_vars="Model",

    value_vars=[

        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"

    ],

    var_name="Metric",

    value_name="Value"

)

plt.figure(figsize=(14,6))

sns.barplot(

    data=metrics_plot,

    x="Model",

    y="Value",

    hue="Metric"

)

plt.title(
    "Executive Model Comparison Dashboard"
)

plt.tight_layout()

plt.show()

In [ ]:
best_model = (

    final_model_comparison
    .sort_values(
        ["Recall", "ROC_AUC"],
        ascending=False
    )
    .head(1)

)

display(best_model.round(4))

In [ ]:
best_rf_params = {

    "n_estimators": 200,
    "max_depth": 10,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1

}

In [ ]:
# =====================================================
# Final Random Forest Model
# =====================================================

from sklearn.ensemble import RandomForestClassifier

final_rf = RandomForestClassifier(
    **best_rf_params
)

final_rf.fit(
    X_train_smote,
    y_train_smote
)

print("Final Random Forest Trained")

In [ ]:
# =====================================================
# Feature Importance Extraction
# =====================================================

rf_importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance":
        final_rf.feature_importances_

})

rf_importance = (

    rf_importance
    .sort_values(
        "Importance",
        ascending=False
    )

)

rf_importance.head(20)

In [ ]:
# =====================================================
# Top 20 Feature Importance
# =====================================================

top_features = rf_importance.head(20)

plt.figure(figsize=(12,8))

sns.barplot(

    data=top_features,

    x="Importance",

    y="Feature",

    palette="viridis"

)

plt.title(
    "Top 20 Chronic Disease Risk Drivers"
)

plt.xlabel(
    "Importance Score"
)

plt.ylabel(
    "Feature"
)

plt.tight_layout()

plt.show()

In [ ]:
# =====================================================
# Cumulative Feature Importance
# =====================================================

rf_importance["Cumulative"] = (
    rf_importance["Importance"]
    .cumsum()
)

plt.figure(figsize=(10,5))

plt.plot(

    range(1, len(rf_importance)+1),

    rf_importance["Cumulative"],

    marker="o"

)

plt.axhline(
    0.80,
    color="red",
    linestyle="--",
    label="80% Importance"
)

plt.title(
    "Cumulative Feature Importance"
)

plt.xlabel(
    "Number of Features"
)

plt.ylabel(
    "Cumulative Importance"
)

plt.legend()

plt.show()

In [ ]:
top_risk_drivers = (
    rf_importance
    .head(10)
    .reset_index(drop=True)
)

top_risk_drivers

In [ ]:
rf_importance.tail(20)

In [ ]:
# =====================================================
# Bottom 20 Least Important Features
# =====================================================

least_important_features = (

    rf_importance
    .sort_values(
        "Importance",
        ascending=True
    )
    .head(20)

)

display(
    least_important_features
)

In [ ]:
# =====================================================
# Least Important Features Visualization
# =====================================================

bottom_features = (

    rf_importance
    .sort_values(
        "Importance",
        ascending=True
    )
    .head(20)

)

plt.figure(figsize=(12,8))

sns.barplot(

    data=bottom_features,

    x="Importance",

    y="Feature",

    palette="Reds"

)

plt.title(
    "Bottom 20 Least Important Features"
)

plt.xlabel(
    "Importance Score"
)

plt.ylabel(
    "Feature"
)

plt.tight_layout()

plt.show()

In [ ]:
# =====================================================
# Features With Very Low Importance
# =====================================================

low_importance_features = (

    rf_importance[
        rf_importance["Importance"] < 0.001
    ]

)

display(
    low_importance_features
)

print(
    "Number of Low Importance Features:",
    len(low_importance_features)
)

In [ ]:
high_risk = member_level_df[
    member_level_df["Develops_Chronic_Disease_12M"] == 1
]

low_risk = member_level_df[
    member_level_df["Develops_Chronic_Disease_12M"] == 0
]

In [ ]:
risk_profile = pd.DataFrame({

    "High Risk Avg": high_risk[[
        "Age",
        "BMI",
        "Glucose",
        "HbA1c",
        "Medications",
        "Chronic_Condition_Count"
    ]].mean(),

    "Low Risk Avg": low_risk[[
        "Age",
        "BMI",
        "Glucose",
        "HbA1c",
        "Medications",
        "Chronic_Condition_Count"
    ]].mean()

})

risk_profile

# MEMBER SEGMENTATION (CLUSTERING)

Can we group members into meaningful population segments?
Instead of Will this member develop disease?

In [ ]:
cluster_features = [

    "Age",
    "BMI",
    "Glucose",
    "HbA1c",
    "Systolic_BP",
    "Diastolic_BP",
    "Medications",
    "Chronic_Condition_Count",
    "Costs",
    "Visits",
    "ER_Visits",
    "Hospital_Admissions",
    "Wellness_Score",
    "Lifestyle_Risk_Score"

]

In [ ]:
from sklearn.preprocessing import StandardScaler

X_cluster = member_level_df[
    cluster_features
].copy()

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X_cluster
)

In [ ]:
# Elbow method to determine optimal number of clusters

from sklearn.cluster import KMeans

inertia = []

for k in range(2,11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)

    inertia.append(
        model.inertia_
    )

plt.figure(figsize=(8,5))

plt.plot(
    range(2,11),
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters")

plt.ylabel("Inertia")

plt.title("Elbow Method")

plt.show()

In [ ]:
# Silhouette Score to determine optimal number of clusters

from sklearn.metrics import silhouette_score

scores = []

for k in range(2,11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(
        X_scaled
    )

    score = silhouette_score(
        X_scaled,
        labels
    )

    scores.append(score)

pd.DataFrame({

    "Clusters": range(2,11),

    "Silhouette_Score": scores

})

In [ ]:
# Cluster creation

optimal_k = 4

kmeans = KMeans(

    n_clusters=optimal_k,

    random_state=42,

    n_init=10

)

member_level_df["Cluster"] = (

    kmeans.fit_predict(
        X_scaled
    )

)

In [ ]:
# Cluster profiling

cluster_profile = (

    member_level_df
    .groupby("Cluster")[

        cluster_features

    ]
    .mean()

)

display(
    cluster_profile.round(2)
)

In [ ]:
member_level_df[
    "Cluster"
].value_counts()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

cluster_pca = pca.fit_transform(
    X_scaled
)

plt.figure(figsize=(10,6))

sns.scatterplot(

    x=cluster_pca[:,0],

    y=cluster_pca[:,1],

    hue=member_level_df["Cluster"],

    palette="tab10"

)

plt.title(
    "Member Segments"
)

plt.show()

# DEEP LEARNING

In [ ]:
# =====================================================
# Deep Learning Dataset Preparation
# =====================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_dl = scaler.fit_transform(
    X_train_smote
)

X_test_dl = scaler.transform(
    X_test
)

print("Train Shape:", X_train_dl.shape)
print("Test Shape :", X_test_dl.shape)

In [ ]:
# =====================================================
# Deep Learning Libraries
# =====================================================

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (

    Dense,
    Dropout,
    BatchNormalization

)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import EarlyStopping

from itertools import product

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score

)

print(tf.__version__)

In [ ]:
import tensorflow as tf

def weighted_binary_crossentropy(
        weight_for_0=1.0,
        weight_for_1=5.0):

    def loss(y_true, y_pred):

        bce = tf.keras.backend.binary_crossentropy(
            y_true,
            y_pred
        )

        weights = (
            y_true * weight_for_1 +
            (1 - y_true) * weight_for_0
        )

        return bce * weights

    return loss

In [ ]:
# =====================================================
# Neural Network Builder
# =====================================================

def build_dl_model(

    hidden_units,
    dropout_rate,
    learning_rate

):

    model = Sequential([

        Dense(

            hidden_units,

            activation="tanh",

            input_shape=(
                X_train_dl.shape[1],
            )

        ),

        BatchNormalization(),

        Dropout(
            dropout_rate
        ),

        Dense(

            hidden_units // 2,

            activation="tanh"

        ),

        Dense(
            1,
            activation="sigmoid"
        )

    ])

    model.compile(

    optimizer="adam",

    loss= weighted_binary_crossentropy(
        weight_for_0=1,
        weight_for_1=5
    ),

    metrics=["accuracy"]

)


    return model

In [ ]:
# =====================================================
# Prevent Overfitting
# =====================================================

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=5,

    restore_best_weights=True

)

In [ ]:
# =====================================================
# Deep Learning Experiments
# =====================================================

units_grid = [

    64,
    128

]

dropout_grid = [

    0.20,
    0.30

]

learning_rate_grid = [

    0.001

]

dl_results = []

In [ ]:
# =====================================================
# Run Experiments
# =====================================================

for units, dropout, lr in product(

    units_grid,
    dropout_grid,
    learning_rate_grid

):

    print(
        f"Running: Units={units}, "
        f"Dropout={dropout}, "
        f"LR={lr}"
    )

    model = build_dl_model(

        hidden_units=units,

        dropout_rate=dropout,

        learning_rate=lr

    )

    history = model.fit(

        X_train_dl,

        y_train_smote,

        validation_split=0.20,

        epochs=30,

        batch_size=32,

        callbacks=[early_stop],

        verbose=0

    )

    y_prob = model.predict(

        X_test_dl,

        verbose=0

    )

    y_pred = (

        y_prob > 0.5

    ).astype(int)

    dl_results.append({

        "Units": units,

        "Dropout": dropout,

        "Learning_Rate": lr,

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Precision":
            precision_score(
                y_test,
                y_pred
            ),

        "Recall":
            recall_score(
                y_test,
                y_pred
            ),

        "F1":
            f1_score(
                y_test,
                y_pred
            ),

        "ROC_AUC":
            roc_auc_score(
                y_test,
                y_prob
            )

    })

In [ ]:
# =====================================================
# Results Table
# =====================================================

dl_results = pd.DataFrame(
    dl_results
)

display(

    dl_results
    .sort_values(
        "Recall",
        ascending=False
    )
    .round(4)

)

In [ ]:
# =====================================================
# Best Neural Network
# =====================================================

best_dl = (

    dl_results
    .sort_values(

        ["Recall", "ROC_AUC"],

        ascending=False

    )

    .head(1)

)

display(
    best_dl.round(4)
)

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    data=dl_results,

    x="Units",

    y="Recall",

    hue="Dropout"

)

plt.title(
    "Deep Learning Recall Comparison"
)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    data=dl_results,

    x="Units",

    y="ROC_AUC",

    hue="Dropout"

)

plt.title(
    "Deep Learning ROC-AUC Comparison"
)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    data=dl_results,

    x="Units",

    y="F1",

    hue="Dropout"

)

plt.title(
    "Deep Learning F1 Comparison"
)

plt.show()

In [ ]:
display(
    dl_results
    .sort_values(
        ["Recall","ROC_AUC"],
        ascending=False
    )
)

In [ ]:
best_dl_final = best_dl.copy()

best_dl_final["Model"] = (
    "Deep Neural Network"
)

best_dl_final = best_dl_final[

    [

        "Model",

        "Accuracy",

        "Precision",

        "Recall",

        "F1",

        "ROC_AUC"

    ]

]

In [ ]:
final_ml_dl_comparison = pd.concat([

    final_model_comparison,

    best_dl_final

], ignore_index=True)

display(
    final_ml_dl_comparison
    .round(4)
)

In [ ]:
comparison_plot = final_ml_dl_comparison.melt(

    id_vars="Model",

    value_vars=[

        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"

    ],

    var_name="Metric",

    value_name="Value"

)

plt.figure(figsize=(14,6))

sns.barplot(

    data=comparison_plot,

    x="Model",

    y="Value",

    hue="Metric"

)

plt.title(
    "Machine Learning vs Deep Learning"
)

plt.tight_layout()

plt.show()

# NLP INSIGHTS
## Objective

Natural Language Processing (NLP) was performed on patient feedback data to understand patient satisfaction, identify service improvement opportunities, discover common themes in patient experiences, and generate actionable healthcare recommendations.

The analysis focuses on:

- Patient sentiment
- Common discussion topics
- Positive experiences
- Negative experiences
- Operational improvement opportunities

In [ ]:
# =====================================================
# Load NLP Dataset
# =====================================================

import pandas as pd

hospital_df = pd.read_csv("hospital.csv")

print("Shape:", hospital_df.shape)

hospital_df.head(20)

In [ ]:
# =====================================================
# Check Available Columns
# =====================================================

hospital_df.columns.tolist()

In [ ]:
# =====================================================
# NLP Dataset Preparation
# =====================================================

import pandas as pd
import numpy as np

nlp_df = hospital_df.copy()

# Remove empty column
nlp_df = nlp_df.drop(
    columns=["Unnamed: 3"],
    errors="ignore"
)

print(nlp_df.shape)

nlp_df.head()

In [ ]:
# =====================================================
# Missing Value Summary
# =====================================================

missing_summary = pd.DataFrame({

    "Missing_Count":
        nlp_df.isnull().sum(),

    "Missing_Percent":
        (
            nlp_df.isnull().mean()*100
        ).round(2)

})

display(missing_summary)

In [ ]:
# =====================================================
# Sentiment Distribution
# =====================================================

sentiment_counts = (
    nlp_df["Sentiment Label"]
    .value_counts()
)

display(sentiment_counts)

In [ ]:
# =====================================================
# Sentiment Distribution Plot
# =====================================================

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))

sns.countplot(

    data=nlp_df,

    x="Sentiment Label"

)

plt.title(
    "Patient Sentiment Distribution"
)

plt.xlabel(
    "Sentiment"
)

plt.ylabel(
    "Review Count"
)

plt.show()

In [ ]:
# =====================================================
# Ratings Distribution
# =====================================================

nlp_df["Ratings"].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(8,4))

sns.countplot(

    data=nlp_df,

    x="Ratings"

)

plt.title(
    "Patient Rating Distribution"
)

plt.show()

## TEXT CLEANING

In [ ]:
# =====================================================
# Text Cleaning
# =====================================================

import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-zA-Z\s]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

nlp_df["Clean_Feedback"] = (

    nlp_df["Feedback"]
    .apply(clean_text)

)

nlp_df[
    [
        "Feedback",
        "Clean_Feedback"
    ]
].head()

In [ ]:
# =====================================================
# TF-IDF
# =====================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(

    stop_words="english",

    max_features=100

)

X_tfidf = tfidf.fit_transform(
    nlp_df["Clean_Feedback"]
)

feature_names = tfidf.get_feature_names_out()

scores = np.asarray(
    X_tfidf.mean(axis=0)
).ravel()

tfidf_df = pd.DataFrame({

    "Word": feature_names,

    "Importance": scores

})

tfidf_df = tfidf_df.sort_values(

    "Importance",

    ascending=False

)

display(
    tfidf_df.head(20)
)

In [ ]:
# =====================================================
# Top Keywords
# =====================================================

top_words = tfidf_df.head(20)

plt.figure(figsize=(12,6))

sns.barplot(

    data=top_words,

    y="Word",

    x="Importance",

    palette="viridis"

)

plt.title(
    "Top Feedback Keywords"
)

plt.show()

In [ ]:
# =====================================================
# Separate Positive and Negative Reviews
# =====================================================

positive_reviews = nlp_df[
    nlp_df["Sentiment Label"] == 1
]

negative_reviews = nlp_df[
    nlp_df["Sentiment Label"] == 0
]

print(
    "Positive:",
    positive_reviews.shape[0]
)

print(
    "Negative:",
    negative_reviews.shape[0]
)

In [ ]:
positive_tfidf = TfidfVectorizer(

    stop_words="english",

    max_features=20

)

positive_matrix = positive_tfidf.fit_transform(

    positive_reviews[
        "Clean_Feedback"
    ]

)

positive_words = pd.DataFrame({

    "Word":
        positive_tfidf
        .get_feature_names_out(),

    "Score":
        np.asarray(
            positive_matrix.mean(axis=0)
        ).ravel()

}).sort_values(

    "Score",

    ascending=False

)

display(positive_words)

In [ ]:
negative_tfidf = TfidfVectorizer(

    stop_words="english",

    max_features=20

)

negative_matrix = negative_tfidf.fit_transform(

    negative_reviews[
        "Clean_Feedback"
    ]

)

negative_words = pd.DataFrame({

    "Word":
        negative_tfidf
        .get_feature_names_out(),

    "Score":
        np.asarray(
            negative_matrix.mean(axis=0)
        ).ravel()

}).sort_values(

    "Score",

    ascending=False

)

display(negative_words)

In [ ]:
# =====================================================
# Topic Modeling
# =====================================================

from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(

    n_components=5,

    random_state=42

)

lda.fit(X_tfidf)

feature_names = tfidf.get_feature_names_out()

for topic_idx, topic in enumerate(
    lda.components_
):

    print(
        f"\nTopic {topic_idx+1}"
    )

    print(

        ", ".join(

            [
                feature_names[i]

                for i in topic.argsort()[-10:]

            ]

        )

    )

In [ ]:
# =====================================================
# Predict Sentiment
# =====================================================

def predict_sentiment(review):

    cleaned_review = clean_text(review)

    vector = tfidf_model.transform(
        [cleaned_review]
    )

    prediction = sentiment_model.predict(
        vector
    )[0]

    probability = sentiment_model.predict_proba(
        vector
    )[0]

    sentiment = (
        "Positive"
        if prediction == 1
        else "Negative"
    )

    print("Review:")
    print(review)

    print("\nPredicted Sentiment:")
    print(sentiment)

    print("\nProbabilities:")
    print(
        {
            "Negative":
            round(probability[0],3),

            "Positive":
            round(probability[1],3)
        }
    )

In [ ]:
test_reviews = [

    "The doctors were excellent and staff was very helpful.",

    "Long waiting time and very poor service.",

    "Billing process was confusing and expensive.",

    "Very satisfied with the treatment and care."

]

In [ ]:
# =====================================================
# Sentiment Classification Dataset
# =====================================================

from sklearn.model_selection import train_test_split

X = nlp_df["Clean_Feedback"]

y = nlp_df["Sentiment Label"]

X_train_nlp, X_test_nlp, y_train_nlp, y_test_nlp = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [ ]:
# =====================================================
# TF-IDF Features
# =====================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_model = TfidfVectorizer(

    stop_words="english",

    max_features=2000

)

X_train_tfidf = tfidf_model.fit_transform(
    X_train_nlp
)

X_test_tfidf = tfidf_model.transform(
    X_test_nlp
)

In [ ]:
# =====================================================
# Sentiment Classification Model
# =====================================================

from sklearn.linear_model import LogisticRegression

sentiment_model = LogisticRegression(
    random_state=42
)

sentiment_model.fit(
    X_train_tfidf,
    y_train_nlp
)

print("Sentiment Model Trained")

In [ ]:
from sklearn.metrics import classification_report

pred = sentiment_model.predict(
    X_test_tfidf
)

print(
    classification_report(
        y_test_nlp,
        pred
    )
)

In [ ]:
negative_words = [

    "bad",
    "worst",
    "rude",
    "delay",
    "waiting",
    "confusing",
    "expensive",
    "poor"

]

def quick_sentiment_check(text):

    text = text.lower()

    for word in negative_words:

        if word in text:
            return "Likely Negative"

    return "Likely Positive"

In [ ]:
# =====================================================
# Predict Sentiment
# =====================================================

def predict_sentiment(review):

    cleaned_review = clean_text(review)

    vector = tfidf_model.transform(
        [cleaned_review]
    )

    prediction = sentiment_model.predict(
        vector
    )[0]

    probability = sentiment_model.predict_proba(
        vector
    )[0]

    sentiment = (
        "Positive"
        if prediction == 1
        else "Negative"
    )

    print("Review:")
    print(review)

    print("\nPredicted Sentiment:")
    print(sentiment)

    print("\nProbabilities:")
    print(
        {
            "Negative":
            round(probability[0],3),

            "Positive":
            round(probability[1],3)
        }
    )

In [ ]:
predict_sentiment(

    "The doctors were excellent and the nurses were very caring."

)

In [ ]:
predict_sentiment(


    

    "Billing process was confusing and expensive.",

    # "Very satisfied with the treatment and care."


)

In [ ]:
def predict_topic(review):

    cleaned = clean_text(review)

    vector = tfidf.transform(
        [cleaned]
    )

    topic_probs = lda.transform(
        vector
    )

    topic = topic_probs.argmax()

    print("Review:")
    print(review)

    print(
        f"\nPredicted Topic: Topic {topic+1}"
    )

    print(
        "\nTopic Distribution:"
    )

    print(topic_probs)

In [ ]:
predict_topic(

    "Waiting time in emergency department was very long."

)

In [ ]:
predict_topic(

    "Billing process was confusing and expensive."

)

In [ ]:
pd.crosstab(
    nlp_df["Sentiment Label"],
    nlp_df["Ratings"]
)

In [ ]:
from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

pred = sentiment_model.predict(
    X_test_tfidf
)

print(
    classification_report(
        y_test_nlp,
        pred
    )
)

accuracy = accuracy_score(
    y_test_nlp,
    pred
)

precision = precision_score(
    y_test_nlp,
    pred
)

recall = recall_score(
    y_test_nlp,
    pred
)

f1 = f1_score(
    y_test_nlp,
    pred
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
cm = confusion_matrix(
    y_test_nlp,
    pred
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title(
    "Sentiment Classification Confusion Matrix"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:
print(member_level_df.shape)

print(member_level_df["Develops_Chronic_Disease_12M"].value_counts())

print(member_level_df["Develops_Chronic_Disease_12M"].value_counts(normalize=True)*100)